In [52]:
import os

import pandas as pd
from pathlib import Path
from graphrag.config.load_config import load_config
import graphrag.api as api

In [2]:
PROJECT_DIRECTORY = "."

# graphrag_config = load_config(Path(PROJECT_DIRECTORY))

entities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/entities.parquet")
communities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/communities.parquet")
text_units = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/text_units.parquet")
relationships = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/relationships.parquet")
reports = pd.read_parquet(
    f"{PROJECT_DIRECTORY}/output/community_reports.parquet"
)

In [49]:
graphrag_config = load_config(Path(PROJECT_DIRECTORY))

In [9]:
query = "A truck driver is injured during an accident.Which policy documents are relevant and why?"

In [54]:
result, context = await api.local_search(
    config=graphrag_config,
    entities=entities,
    relationships=relationships,
    text_units=text_units,
    communities=communities,
    community_reports=reports,
    community_level=2,
    covariates=None,
    response_type="Multiple Paragraphs",
    query=query,
)

In [53]:
for k in context.keys():
    # print(context[k])
    print(len(context[k]), " - ", k)

2  -  reports
29  -  relationships
0  -  claims
13  -  entities
6  -  sources


In [34]:
print(context["entities"].columns)

print(context["relationships"].columns)

print(context["sources"].columns)

Index(['id', 'entity', 'description', 'number of relationships', 'in_context'], dtype='object')
Index(['id', 'source', 'target', 'description', 'weight', 'links',
       'in_context'],
      dtype='object')
Index(['id', 'text'], dtype='object')


In [40]:
entity_set = set(context["entities"]["entity"])

missing_nodes = set()
for _, row in context["relationships"].iterrows():

    if row["source"] not in entity_set:
        missing_nodes.add(row["source"])

    if row["target"] not in entity_set:
        missing_nodes.add(row["target"])

print(missing_nodes)

{'IRDAI', 'NETWORK GARAGE', 'INSURANCE OMBUDSMAN', 'INSURANCE REGULATORY AND DEVELOPMENT AUTHORITY OF INDIA (IRDAI)', 'WEBSITE', 'POLLUTION UNDER CONTROL CERTIFICATE', 'ICICI LOMBARD GENERAL INSURANCE COMPANY LIMITED'}


In [37]:
context["entities"]

,id,entity,description,number of relationships,in_context
0,28,POLICY,The insurance policy is the contract detailing...,1,True
1,25,COMPANY,Refers to the unnamed entity providing the ins...,5,True
2,39,INSURED VEHICLE,An insured vehicle is any vehicle that has an ...,2,True
3,37,PART 1 OF THE SCHEDULE,Part 1 of the Schedule refers to the section o...,1,True
4,26,INSURED,"The term ""insured"" refers to the individual or...",8,True
5,18,GRIEVANCE REDRESSAL,Grievance redressal encompasses the structured...,3,True
6,15,ICICI LOMBARD,ICICI Lombard is a prominent general insurance...,11,True
7,29,POLLUTION UNDER CONTROL (PUC) CERTIFICATE,The PUC Certificate is a necessary document ve...,1,True
8,24,IL SMART ASSIST ADD-ON,IL Smart Assist Add-On is an insurance service...,1,True
9,30,FITNESS CERTIFICATE,The Fitness Certificate is a crucial verificat...,2,True


In [38]:
context["relationships"]

,id,source,target,description,weight,links,in_context
0,33,INSURED,ICICI LOMBARD,The insured is an individual or entity that ho...,9.0,3,True
1,44,ICICI LOMBARD,BIMA BHAROSA PORTAL,ICICI Lombard customers can access IRDAI servi...,1.0,3,True
2,6,ICICI LOMBARD,INSURANCE REGULATORY AND DEVELOPMENT AUTHORITY...,ICICI Lombard is an insurance company that ope...,16.0,1,True
3,7,ICICI LOMBARD,GRIEVANCE REDRESSAL,ICICI Lombard offers a dedicated grievance red...,13.0,3,True
4,36,ICICI LOMBARD,CUSTOMER SUPPORT,ICICI Lombard offers customer support services...,8.0,2,True
5,12,INSURED,COMPANY,"The insured holds a policy with the company, r...",9.0,2,True
6,38,FITNESS CERTIFICATE,ICICI LOMBARD,ICICI Lombard stipulates that insured vehicle ...,7.0,2,True
7,41,INSURED VEHICLE,ICICI LOMBARD,The insured vehicle is covered by ICICI Lombar...,8.0,3,True
8,14,INSURED,BIMA BHAROSA PORTAL,The insured can use the Bima Bharosa Portal fo...,6.0,3,True
9,39,BIMA BHAROSA PORTAL,INSURED,Insured individuals may use the Bima Bharosa P...,1.0,3,True


<h2>Graph path highlighted</h2>

In [48]:
from pyvis.network import Network

# --------------------------
# Retrieved nodes/edges
# --------------------------

retrieved_entities = set(
    context["entities"]["entity"].astype(str)
)

retrieved_edges = set()

for _, row in context["relationships"].iterrows():
    retrieved_edges.add(
        (str(row["source"]), str(row["target"]))
    )

# --------------------------
# Entity descriptions
# --------------------------

entity_desc = {}

for _, row in entities.iterrows():

    entity_name = str(
        row.get("title", row.get("entity", ""))
    )

    entity_desc[entity_name] = str(
        row.get("description", "")
    )

# --------------------------
# Create graph
# --------------------------

net = Network(
    height="900px",
    width="100%",
    directed=True,
    bgcolor="#ffffff",
    font_color="black"
)

# --------------------------
# Collect all nodes
# --------------------------

all_nodes = set()

for _, row in relationships.iterrows():

    source = str(row["source"])
    target = str(row["target"])

    all_nodes.add(source)
    all_nodes.add(target)

# --------------------------
# Add nodes
# Red = retrieved
# Grey = not retrieved
# --------------------------

for node in all_nodes:

    if node in retrieved_entities:

        net.add_node(
            node,
            label=node,
            title=entity_desc.get(node, ""),
            color="#ff4444",
            size=20
        )

    else:

        net.add_node(
            node,
            label=node,
            title=entity_desc.get(node, ""),
            color="#d3d3d3",
            size=20
        )

# --------------------------
# Add edges
# Red = retrieved
# Grey = not retrieved
# --------------------------

for _, row in relationships.iterrows():

    source = str(row["source"])
    target = str(row["target"])

    edge = (source, target)

    if edge in retrieved_edges:

        net.add_edge(
            source,
            target,
            color="red",
            width=2,
            title=str(row.get("description", ""))
        )

    else:

        net.add_edge(
            source,
            target,
            color="#dddddd",
            width=1,
            title=str(row.get("description", ""))
        )

# --------------------------
# Physics
# --------------------------

net.force_atlas_2based()

# Optional:
# net.show_buttons(filter_=["physics"])

# --------------------------
# Save
# --------------------------

net.write_html(
    "graphrag_retrieval_path.html",
    open_browser=False
)

print("Saved: graphrag_retrieval_path.html")

Saved: graphrag_retrieval_path.html


In [56]:
print(result)

### Relevant Policy Documents for Truck Driver Injury

In the event of a truck driver getting injured during an accident, several key policy documents from ICICI Lombard would be relevant for understanding the coverage and handling of the incident. These documents detail the terms of the insurance policy, define the coverage, and establish procedures for claims.

#### 1. **Insurance Policy Document**
The primary document in consideration is the insurance policy itself, which outlines the specifics of coverage for accidents, including:

- **Terms and Conditions**: It details conditions related to the policyholder's eligibility for coverage in the event of injuries sustained during an accident. This contract is crucial as it stipulates the rights and obligations of both the insured (the truck owner/driver) and the insurer [Data: Entities (28), 25)].

#### 2. **Grievance Redressal Mechanism**
If the driver faces issues regarding claims or support following the injury, the grievance redres

In [79]:
retrieved_docs = set()

for _, source_row in context["sources"].iterrows():

    source_text = source_row["text"]

    matches = text_units[
        text_units["text"] == source_text
    ]

    if not matches.empty:

        retrieved_docs.update(
            matches["document_id"].tolist()
        )

print(retrieved_docs)
print(len(retrieved_docs))
documents = pd.read_parquet(
    "output/documents.parquet"
)

# print(documents.head())
documents[
    documents["id"].isin(retrieved_docs)
][["id", "title"]]

{'1fc39c03e090c7e32ab65220b39d9bf8d1f0c51d1f75f9ba3e4d670a163896084344b6bc83c31c8e128ec9f2f567f618f33b0e030310ab2b5ad15d58eecd0534', '520c9a587083dde8eb812ec64ad45167bc9d183489b34da69ded2725d30b82a21d23b3e8a9fd6847723a0a9ef7078721673e18b202a5debce74e33d22f674302', '2e2e5881a10616f2f4cf683318526259ce91966ce8181dec9ef2fc1a06da24c2621e56266565cc78534be97cd1200718e863e2f274eda24949f032c28ac5ea63', 'd81ec8d26068e6fe615233e8a1a3fedf1d150c5d3efbcad4e00f446fe4446c127b8304acf57240cd85839f02b97058b9fa0e0a48d95dd89adcf844c19d08ec2f'}
4


,id,title
1,520c9a587083dde8eb812ec64ad45167bc9d183489b34d...,compulsory-personal-accident-owner-driver-poli...
2,1fc39c03e090c7e32ab65220b39d9bf8d1f0c51d1f75f9...,3-years-private-car-liability-policy.txt
3,d81ec8d26068e6fe615233e8a1a3fedf1d150c5d3efbca...,personal-protect-policy_policy-wordings.txt
4,2e2e5881a10616f2f4cf683318526259ce91966ce8181d...,goods-carrying-vehicle-liability-policy-wordin...
